In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:86% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:12pt;}
div.output {font-size:15pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:15px;}
</style>
"""))

<font size=5, color='red'>ch03. 연관분석</font>
- pip install apyori
# 1. 연관 분석 개요
- 데이터들 사이에 자주 발생하는 속성을 찾고 그 속성들 사이에 연관성이 어느 정도 있는 지 분석
- 활용분야 : 이벤트 미리 감지(사기 적발..), 신상품 카테고리 구성
- 용어사전
    예시와 함께 설명, 보조자료 'data/cf_basket.csv' 체크
    [조건 : left hand side as lhs] 오렌지주스 => [결과 : right hand side as rhs] 와인
    - 연관분석과 관련된 지표
    1. 지지도(support) : 얼마나 자주 함께 나타나는 지
        (lhs와 rhs)의 항목수 / 전체 항목수 = 0.2
        
    2. 신뢰도(confidence) : 조건이 오면 결과가 얼마나 자주 나타나는 지
        (lhs -> rhs)의 항목수 / lhs가 포함된 항목수 = 0.5
        
    3. 향상도(lift) : 우연히 발생한 규칙은 아닌 지 확인
        (lhs -> rhs)의 지지도 / (lhs의 지지도) * (rhs의 지지도)
            => 0.2 / (0.4 * 0.6) = 0.2 / 0.24 =0.8333...
            향상도 < 1 기대가 낮음(없음)
            향상도 > 1 기대가 높음(있음)

# 2. 연관 분석 구현

In [7]:
import csv
transaction = []
with open('data/cf_basket.csv','r',encoding='utf-8') as f :
    csvdata = csv.reader(f)
    transaction = list(csvdata)
transaction

[['소주', '콜라', '와인'],
 ['소주', '오렌지주스', '콜라'],
 ['맥주', '콜라', '와인'],
 ['소주', '콜라', '맥주'],
 ['오렌지주스', '와인']]

In [12]:
from apyori import apriori
rules = apriori(transaction, # 2차원 데이터
                min_support = 0.15,
               min_confidence = 0.1,
               ) # 매개변수 : min_support min_confidence min_lift max_length 
# len(list(rules)) # 18개 규칙이 생성
rules = list(rules)

In [54]:
rules[10][2]

[OrderedStatistic(items_base=frozenset(), items_add=frozenset({'소주', '콜라'}), confidence=0.6, lift=1.0),
 OrderedStatistic(items_base=frozenset({'소주'}), items_add=frozenset({'콜라'}), confidence=1.0, lift=1.25),
 OrderedStatistic(items_base=frozenset({'콜라'}), items_add=frozenset({'소주'}), confidence=0.7499999999999999, lift=1.2499999999999998)]

In [46]:
rule = rules[10]
support = rule[1]
order_st = rule[2]
print('{LHS}\t\t => \t\t{RHS}\t     {support} {confidence} {lift}')
for item in order_st :
    lhs = item[0]
    rhs = item[1]
    confidence = item[2]
    lift = item[3]
    if lift > 1 :
        print('{} => {}\t {:.2f} \t  {:.2f} \t     {:.2f}'.format(lhs, rhs, support, confidence, lift))

{LHS}		 => 		{RHS}	     {support} {confidence} {lift}
frozenset({'소주'}) => frozenset({'콜라'})	 0.60 	  1.00 	     1.25
frozenset({'콜라'}) => frozenset({'소주'})	 0.60 	  0.75 	     1.25


In [47]:
for rule in rules :
    support = rule[1]
    order_st = rule[2]
    
    for item in order_st :
        lhs = item[0]
        rhs = item[1]
        confidence = item[2]
        lift = item[3]
        if lift > 1 :
            print('{} => {}\t {:.2f} \t  {:.2f} \t     {:.2f}'.format(lhs, rhs, support, confidence, lift))

frozenset({'맥주'}) => frozenset({'콜라'})	 0.40 	  1.00 	     1.25
frozenset({'콜라'}) => frozenset({'맥주'})	 0.40 	  0.50 	     1.25
frozenset({'소주'}) => frozenset({'콜라'})	 0.60 	  1.00 	     1.25
frozenset({'콜라'}) => frozenset({'소주'})	 0.60 	  0.75 	     1.25
frozenset({'콜라'}) => frozenset({'소주', '맥주'})	 0.20 	  0.25 	     1.25
frozenset({'소주', '맥주'}) => frozenset({'콜라'})	 0.20 	  1.00 	     1.25
frozenset({'맥주'}) => frozenset({'콜라', '와인'})	 0.20 	  0.50 	     1.25
frozenset({'콜라'}) => frozenset({'맥주', '와인'})	 0.20 	  0.25 	     1.25
frozenset({'와인', '맥주'}) => frozenset({'콜라'})	 0.20 	  1.00 	     1.25
frozenset({'콜라', '와인'}) => frozenset({'맥주'})	 0.20 	  0.50 	     1.25
frozenset({'소주'}) => frozenset({'콜라', '오렌지주스'})	 0.20 	  0.33 	     1.67
frozenset({'콜라'}) => frozenset({'소주', '오렌지주스'})	 0.20 	  0.25 	     1.25
frozenset({'소주', '오렌지주스'}) => frozenset({'콜라'})	 0.20 	  1.00 	     1.25
frozenset({'콜라', '오렌지주스'}) => frozenset({'소주'})	 0.20 	  1.00 	     1.67
frozenset({'콜라'}) => frozenset({

In [66]:
import pandas as pd
rules_df = pd.DataFrame(None, columns=['lhs','rhs','지지도','신뢰도','향상도'])
# rules_df.loc[0] = ['와인','오렌지',0.15,0.5,1.11]
i = 0
for rule in rules :
    support = rule[1]
    order_st = rule[2]
    
    for item in order_st :
        lhs = [i for i in item[0]]
        rhs = [i for i in item[1]]
        confidence = item[2]
        lift = item[3]
        if lift > 1 :
            rules_df.loc[i] = [' '.join(lhs),' '.join(rhs),support,confidence,lift]
            i +=1
rules_df.sort_values(by=['향상도','신뢰도'], ascending=False)

,lhs,rhs,지지도,신뢰도,향상도
13,콜라 오렌지주스,소주,0.2,1.000000,1.666667
10,소주,콜라 오렌지주스,0.2,0.333333,1.666667
0,맥주,콜라,0.4,1.000000,1.250000
2,소주,콜라,0.6,1.000000,1.250000
5,소주 맥주,콜라,0.2,1.000000,1.250000
8,와인 맥주,콜라,0.2,1.000000,1.250000
12,소주 오렌지주스,콜라,0.2,1.000000,1.250000
15,소주 와인,콜라,0.2,1.000000,1.250000
1,콜라,맥주,0.4,0.500000,1.250000
6,맥주,콜라 와인,0.2,0.500000,1.250000


# 3. 경주/전주 여행 자료 연관 분석

In [77]:
df = pd.read_csv('data/naver_kin.csv', sep = '\t')
total_text_list = df['total_text'].tolist()

In [91]:
%%time
# 이 셀은 실행시킬 때 시간 오래 걸리니까 주의
from konlpy.tag import Hannanum, Kkma
hannanum = Kkma()
total_noun_list = []
select_pos = {'NC','NQ', 'NNP','NNG'}
불용어 = {'여행', '전주여행', '경주여행'}
for text in total_text_list :
    temp=[word for word, tag in hannanum.pos(text) if tag in select_pos and word not in 불용어 and len(word) > 1 ]# 보통명사, 고유명사만 추출
    total_noun_list.append(temp)

In [92]:
total_noun_list

[['전주',
  '추천',
  '추억',
  '다양',
  '체험',
  '추억',
  '테마',
  '박물관',
  '유익',
  '시간',
  '투어',
  '패스',
  '통합',
  '이용권',
  '여행지',
  '다양',
  '체험',
  '카페',
  '이용',
  '추가',
  '전주',
  '필수',
  '편안',
  '감사'],
 ['전주',
  '전주',
  '사람',
  '호텔',
  '가격',
  '얼마',
  '정도',
  '카페',
  '추천',
  '주세',
  '안녕',
  '하세',
  '전주',
  '계획',
  '중이',
  '한옥',
  '마을',
  '근처'],
 ['전주',
  '관련',
  '질문',
  '전주',
  '계획',
  '선택',
  '전주',
  '한옥',
  '마을',
  '자연',
  '경관',
  '음식',
  '동네',
  '시간',
  '한옥',
  '마을',
  '근처',
  '장소'],
 ['학생', '친구', '전주', '안녕', '하세', '여학생', '친구', '전주', '가기', '일본', '전주', '오타쿠'],
 ['모닝',
  '전주',
  '부모님',
  '전주',
  '명소',
  '식당',
  '추천',
  '전주',
  '한옥',
  '마을',
  '유명',
  '완산구',
  '덕진구',
  '추억',
  '장소'],
 ['전주',
  '이번',
  '여자',
  '친구',
  '전주',
  '경로',
  '주세',
  '토요일',
  '실내',
  '데이트',
  '가능',
  '감사',
  '숙소',
  '전주',
  '한옥',
  '마을',
  '근처',
  '한옥',
  '마을',
  '구경'],
 ['전주',
  '저녁',
  '전주',
  '산책길',
  '한옥',
  '마을',
  '근처',
  '야경',
  '일정',
  '관심',
  '테마',
  '사진',
  '카페',
  '체험',
  '맞춤',
  '추천',
  '전주'],
 ['전주',
  '코

In [103]:
%%time
from apyori import apriori
rules = apriori(total_noun_list, # 2차원 데이터
                min_support = 0.15,
               min_confidence = 0.3,
               ) # 매개변수 : min_support min_confidence min_lift max_length 
rules = list(rules)

import pandas as pd
rules_df = pd.DataFrame(None, columns=['lhs','rhs','지지도','신뢰도','향상도'])

i = 0
for rule in rules :
    support = rule[1]
    order_st = rule[2]
    
    for item in order_st :
        lhs = [i for i in item[0]]
        rhs = [i for i in item[1]]
        confidence = item[2]
        lift = item[3]
        if lift > 1 :
            rules_df.loc[i] = [' '.join(lhs),' '.join(rhs),round(support,2),round(confidence,2),round(lift,2)]
            i +=1
rules_df.sort_values(by=['향상도','신뢰도','지지도'], ascending=False, inplace=True)
rules_df = rules_df.reset_index(drop=True)

CPU times: total: 5.27 s
Wall time: 5.26 s


In [104]:
# pd.options.display.max_rows
rules_df[:60]

,lhs,rhs,지지도,신뢰도,향상도
0,볼만 국내,주가 여수,0.16,1.0,5.99
1,주가 국내,볼만 여수,0.16,1.0,5.99
2,볼만 국내,여수 코스,0.16,1.0,5.99
3,볼만 국내,여수 티비,0.16,1.0,5.99
4,주가 국내,여수 코스,0.16,1.0,5.99
5,주가 국내,여수 티비,0.16,1.0,5.99
6,볼만 국내,전주 주가 여수,0.16,1.0,5.99
7,주가 국내,전주 볼만 여수,0.16,1.0,5.99
8,전주 볼만 국내,주가 여수,0.16,1.0,5.99
9,전주 주가 국내,볼만 여수,0.16,1.0,5.99
